## Project Objective
In this notebook, we perform feature engineering on the cleaned RetailRocket e-commerce dataset.

The primary objective of this notebook is to:

- Transform raw interactions into machine learning-ready features
- Build user-level behavioral features
- Build item-level popularity and engagement features
- Prepare structured input for recommendation algorithms
- Align dataset with modular src/ pipeline scripts

This notebook strictly focuses on feature engineering only and follows a modular ML architecture where core logic is also implemented in reusable .py scripts inside the src/ folder.

In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Load cleaned datasets
events_df = pd.read_csv('../data/processed/cleaned_events.csv')

In [ ]:
events_df['datetime'] = pd.to_datetime(events_df['timestamp'])

In [ ]:
#Convert categorical interaction types into numerical signals
event_mapping = {
    'view': 1,
    'addtocart': 2,
    'transaction': 3
}
events_df['event_score'] = events_df['event'].map(event_mapping)

In [ ]:
#Total Interactions per User
user_interactions = events_df.groupby('visitorid').size().reset_index(name='total_interactions')

In [ ]:
user_event_counts = events_df.pivot_table(
    index='visitorid',
    columns='event',
    aggfunc='size',
    fill_value=0
).reset_index()

In [ ]:
user_purchases = events_df[events_df['event'] == 'transaction']\
    .groupby('visitorid')\
    .size()\
    .reset_index(name='total_purchases')

In [ ]:
# tells popularity of item 
item_popularity = events_df.groupby('itemid').size().reset_index(name='interaction_count')

In [ ]:
item_purchases = events_df[events_df['event'] == 'transaction']\
    .groupby('itemid')\
    .size()\
    .reset_index(name='purchase_count') 

In [ ]:
events_df['hour'] = events_df['datetime'].dt.hour

In [ ]:
hourly_activity = events_df.groupby('hour').size().reset_index(name='activity_count')

In [ ]:
max_date = events_df['datetime'].max()   # recent date 
events_df['recency_days'] = (max_date - events_df['datetime']).dt.days

In [ ]:
event_weight = {
    'view': 1,
    'addtocart': 3,
    'transaction': 5
}
events_df['interaction_strength'] = events_df['event'].map(event_weight)

In [ ]:
final_features = events_df.groupby(['visitorid', 'itemid']).agg({
    'interaction_strength': 'sum',
    'recency_days': 'min'
}).reset_index()

In [ ]:
final_features

In [ ]:
final_features.to_csv('../data/processed/engineered_features.csv', index=False)

## Conclusion

In this notebook, we successfully engineered features for recommendation modeling:

- User-level behavioral features
- Item-level popularity features
- Interaction strength scoring
- Recency-based signals

These features will be used for:

Collaborative filtering
Ranking models
Personalization systems

This feature engineering pipeline ensures structured and scalable ML model development.